In [1]:
# Cell 1: Setup and imports
import pandas as pd
import os
import numpy as np

# Define paths
base_path = os.path.dirname(os.path.dirname(os.path.dirname(os.getcwd()))) +"/Scrutins/Scrutins"

path14 = os.path.join(base_path, "1_data_extraction/14e_legislature_20juin2012-20juin2017/1_extract_python")
path15 = os.path.join(base_path, "1_data_extraction/15e_legislature_27juin2017-21juin2022/1_extract_python")
path16 = os.path.join(base_path, "1_data_extraction/16e_legislature_ 28juin2022-9juin2024/1_extract_python")
path17 = os.path.join(base_path, "1_data_extraction/17e_legislature_18juillet2024/1_extract_python")
path2 = os.path.join(base_path, "2_cleaning_stata")
path3 = os.path.join(base_path, "3_dataset")

# Create directories if needed
os.makedirs(path2, exist_ok=True)
os.makedirs(path3, exist_ok=True)

print(f"Paths configured:\n 14e: {path14}\n 15e: {path15}\n 16e: {path16}\n 17e: {path17}\n Output: {path3}")

Paths configured:
 14e: c:\Users\rorod\Documents\Pro\RDB Consulting\Mission Hélène 11-2025--02-2026\DIMIG (collab with Hélène)\AN/Scrutins/Scrutins\1_data_extraction/14e_legislature_20juin2012-20juin2017/1_extract_python
 15e: c:\Users\rorod\Documents\Pro\RDB Consulting\Mission Hélène 11-2025--02-2026\DIMIG (collab with Hélène)\AN/Scrutins/Scrutins\1_data_extraction/15e_legislature_27juin2017-21juin2022/1_extract_python
 16e: c:\Users\rorod\Documents\Pro\RDB Consulting\Mission Hélène 11-2025--02-2026\DIMIG (collab with Hélène)\AN/Scrutins/Scrutins\1_data_extraction/16e_legislature_ 28juin2022-9juin2024/1_extract_python
 17e: c:\Users\rorod\Documents\Pro\RDB Consulting\Mission Hélène 11-2025--02-2026\DIMIG (collab with Hélène)\AN/Scrutins/Scrutins\1_data_extraction/17e_legislature_18juillet2024/1_extract_python
 Output: c:\Users\rorod\Documents\Pro\RDB Consulting\Mission Hélène 11-2025--02-2026\DIMIG (collab with Hélène)\AN/Scrutins/Scrutins\3_dataset


In [2]:
# Cell 2: Define cleaning function
def clean_scrutins(df):
    """
    Clean and rename scrutins dataframe
    Equivalent to the Stata 'cleaning' program in 'build_dataset.do'
    """
    
    # Drop columns that are not needed in analysis
    cols_to_drop = [
        'votant/causePositionVote',
        'main/numero',
        'votant/parDelegation',
        'main/organeRef',
    ]
    df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])
    
    # IDs and core variables with exact headers from scrutins.csv
    rename_dict = {
        'main/uid': 'scrutin_id',
        'votant/acteurRef': 'acteur_id',
        'votant/mandatRef': 'mandat_id',
        'main/seanceRef': 'seance_id',
        'groupe/organeRef': 'groupe_id',
        
        # Scrutin variables
        'main/legislature': 'scrutin_legislature',
        'main/dateScrutin': 'scrutin_date',
        'main/quantiemeJourSeance': 'scrutin_jour_seance',
        'main/typeVote/codeTypeVote': 'scrutin_type_code',
        'main/typeVote/typeMajorité': 'scrutin_type_majorite',  # UTF-8 accent
        'main/typeVote/typeMajoritÃ©': 'scrutin_type_majorite',   # Windows-1252 decoded as UTF-8
        'main/sort/code': 'scrutin_resultat',
        'main/titre': 'scrutin_titre',
        'main/objet/libelle': 'scrutin_objet',
        'main/syntheseVote/nombreVotants': 'scrutin_nb_votants',
        'main/syntheseVote/suffragesExprimes': 'scrutin_nb_suffrages',
        'main/syntheseVote/nbrSuffragesRequis': 'scrutin_nb_requis',
        'main/syntheseVote/annonce': 'scrutin_annonce',
        'main/syntheseVote/decompte/nonVotants': 'scrutin_nb_nonvotants',
        'main/syntheseVote/decompte/pour': 'scrutin_nb_pour',
        'main/syntheseVote/decompte/contre': 'scrutin_nb_contre',
        'main/syntheseVote/decompte/abstentions': 'scrutin_nb_abs',
        'main/syntheseVote/decompte/nonVotantsVolontaires': 'scrutin_nb_volnonvotant',
        
        # Groupe variables
        'groupe/nombreMembresGroupe': 'groupe_nb_membres',
        'groupe/vote/positionMajoritaire': 'groupe_pos_maj',
        'groupe/vote/decompteVoix/nonVotants': 'groupe_nb_nonvotant',
        'groupe/vote/decompteVoix/pour': 'groupe_nb_pour',
        'groupe/vote/decompteVoix/contre': 'groupe_nb_contre',
        'groupe/vote/decompteVoix/abstentions': 'groupe_nb_abs',
        'groupe/vote/decompteVoix/nonVotantsVolontaires': 'groupe_nb_volnonvotant',
        
        # Acteur variables
        'votant/vote': 'acteur_vote',
    }
    
    # Rename columns that exist
    df = df.rename(columns={k: v for k, v in rename_dict.items() if k in df.columns})
    
    # Clean acteur_vote values
    if 'acteur_vote' in df.columns:
        df['acteur_vote'] = df['acteur_vote'].replace({
            'pours': 'pour',
            'contres': 'contre',
            'nonVotants': 'non votant',
            'abstentions': 'abstention',
        })
    
    # Reorder columns: IDs first
    id_cols = [col for col in ['scrutin_id', 'acteur_id', 'mandat_id', 'seance_id', 'groupe_id'] if col in df.columns]
    other_cols = [col for col in df.columns if col not in id_cols]
    df = df[id_cols + other_cols]
    
    return df

print("Cleaning function defined")

Cleaning function defined


In [3]:
# Cell 3: Load and clean 14th legislature
df_14 = pd.read_csv(os.path.join(path14, "scrutins.csv"))
print(f"Loaded 14th legislature: {df_14.shape}")

df_14 = clean_scrutins(df_14)
print(f"After cleaning: {df_14.shape}")

Loaded 14th legislature: (109913, 33)
After cleaning: (109913, 30)


In [4]:
# Cell 3: Load and clean 15th legislature
df_15 = pd.read_csv(os.path.join(path15, "scrutins.csv"))
print(f"Loaded 15th legislature: {df_15.shape}")

df_15 = clean_scrutins(df_15)
print(f"After cleaning: {df_15.shape}")

Loaded 15th legislature: (472631, 34)
After cleaning: (472631, 30)


In [5]:
# Cell 4: Load and clean 16th legislature
df_16 = pd.read_csv(os.path.join(path16, "scrutins.csv"))
print(f"Loaded 16th legislature: {df_16.shape}")

df_16 = clean_scrutins(df_16)
print(f"After cleaning: {df_16.shape}")

Loaded 16th legislature: (603813, 34)
After cleaning: (603813, 30)


In [6]:
# Cell 4b: Load and clean 17th legislature (reduced schema)
df_17 = pd.read_csv(os.path.join(path17, "scrutins.csv"))
print(f"Loaded 17th legislature: {df_17.shape}")

df_17 = clean_scrutins(df_17)
print(f"After cleaning: {df_17.shape}")

Loaded 17th legislature: (856774, 34)
After cleaning: (856774, 30)


In [7]:
# Cell 5: Append all legislatures (14e, 15e, 16e, 17e)
df_scrutins = pd.concat([df_14, df_15, df_16, df_17], ignore_index=True)
print(f"Appended: {df_scrutins.shape}")

# Save merged
df_scrutins.to_csv(os.path.join(path3, "scrutins.csv"), index=False)
print(f"Saved merged to {path3}/scrutins.csv")

Appended: (2043131, 30)
Saved merged to c:\Users\rorod\Documents\Pro\RDB Consulting\Mission Hélène 11-2025--02-2026\DIMIG (collab with Hélène)\AN/Scrutins/Scrutins\3_dataset/scrutins.csv


In [8]:
# Cell 6: Load and merge with organes
# Load organes from Députés dataset
organes_path = os.path.dirname(os.path.dirname(os.path.dirname(os.getcwd()))) + "/Députés_français (Locuteurs)/2_cleaning_stata"

try:
    # Try loading from csv file first
    df_organes = pd.read_csv(os.path.join(organes_path, "organes.csv"))
except:
    # Fallback to stata
    df_organes = pd.read_stata(os.path.join(organes_path, "organes.dta"))

    

print(f"Loaded organes: {df_organes.shape}")
print(f"Organes columns: {df_organes.columns.tolist()}")

Loaded organes: (10752, 4)
Organes columns: ['organe_id', 'codeType', 'libelleAbrege', 'organe_libelle_ANO']


In [9]:
# Cell 7: Rename organes columns and merge
# Rename organe columns before merge
# Actual columns in df_organes: 'organe_id', 'codeType', 'libelleAbrege', 'organe_libelle_ANO'
df_organes = df_organes.rename(columns={
    'organe_id': 'groupe_id',
    'codeType': 'groupe_codetype_ANO',
    'libelleAbrege': 'groupe_abrege_ANO',
    'organe_libelle_ANO': 'groupe_libelle_ANO'
})

# Merge scrutins with organes on groupe_id
df_merged = df_scrutins.merge(df_organes, on='groupe_id', how='left')
print(f"After merge: {df_merged.shape}")

# Check merge quality
if 'groupe_abrege_ANO' in df_merged.columns:
    print(f"Rows with missing groupe info: {df_merged['groupe_abrege_ANO'].isna().sum()}")
else:
    print("Warning: colonne groupe_abrege_ANO absente après merge")

# Reorder columns: IDs first
id_cols = [col for col in ['scrutin_id', 'acteur_id', 'mandat_id', 'seance_id', 'groupe_id'] 
           if col in df_merged.columns]
actor_cols = [col for col in df_merged.columns if col.startswith('acteur_')]
scrutin_cols = [col for col in df_merged.columns if col.startswith('scrutin_')]
groupe_cols = [col for col in df_merged.columns if col.startswith('groupe_')]
other_cols = [col for col in df_merged.columns 
              if col not in id_cols + actor_cols + scrutin_cols + groupe_cols]

df_final = df_merged[id_cols + actor_cols + scrutin_cols + groupe_cols + other_cols]
print(f"Reordered columns: {df_final.shape}")

After merge: (2043131, 33)
Rows with missing groupe info: 0
Reordered columns: (2043131, 36)


In [10]:
# Cell 9: Load and merge with mandats (députés)
import re

# Load mandats from Députés dataset - use FINAL version (3_dataset) with groupe info merged
mandats_path = os.path.dirname(os.path.dirname(os.path.dirname(os.getcwd()))) + "/Députés_français (Locuteurs)/3_dataset"

try:
    # Try loading from csv file first
    df_mandats = pd.read_csv(os.path.join(mandats_path, "mandats.csv"))
except:
    # Fallback to stata
    df_mandats = pd.read_stata(os.path.join(mandats_path, "mandats.dta"))

print(f"Loaded mandats (FINAL version with groupe info): {df_mandats.shape}")
print(f"Mandats columns (first 15): {df_mandats.columns.tolist()[:15]}")

# Normaliser les identifiants PA (PA + chiffres, padding sur 4) pour comparer
def normalize_pa(series):
    if isinstance(series, pd.DataFrame):
        series = series.iloc[:, 0]
    def norm_one(x):
        if pd.isna(x):
            return np.nan
        x = str(x).strip().upper()
        if x.startswith('PA'):
            digits = re.sub(r"[^0-9]", "", x[2:])
            digits = digits.lstrip('0')
            return 'PA' + digits.zfill(4)
        return x
    return series.apply(norm_one)

# Normaliser côté scrutins (left)
left_ids_norm = normalize_pa(df_final['acteur_id'])
df_final['acteur_id_merge'] = left_ids_norm

# Chercher la colonne mandats qui recouvre le plus les ids PA des scrutins
candidate_stats = []
for col in df_mandats.columns:
    s = df_mandats[col]
    if isinstance(s, pd.DataFrame):
        s = s.iloc[:, 0]
    if not pd.api.types.is_object_dtype(s):
        continue
    right_norm = normalize_pa(s)
    inter = len(set(left_ids_norm.dropna()) & set(right_norm.dropna()))
    coverage = inter / max(1, len(left_ids_norm.dropna()))
    candidate_stats.append((col, inter, coverage))

candidate_stats = sorted(candidate_stats, key=lambda x: x[1], reverse=True)
best_col = None
best_inter = 0
if candidate_stats:
    print("Top chevauchements acteur_id (mandats vs scrutins):")
    for col, inter, cov in candidate_stats[:5]:
        print(f"  {col}: intersections {inter}, couverture {cov:.4f}")
    best_col, best_inter, best_cov = candidate_stats[0]
    if best_inter == 0:
        print("Alerte: aucune colonne ne chevauche les ids PA des scrutins.")
        best_col = None
else:
    print("Aucune colonne candidate objet trouvée dans df_mandats.")

if best_col:
    print(f"Colonne retenue pour acteur_id_merge: {best_col} (intersections {best_inter})")
    df_mandats['acteur_id_merge'] = normalize_pa(df_mandats[best_col])
else:
    df_mandats['acteur_id_merge'] = np.nan

# Select relevant columns for merge (key = acteur_id_merge)
mandats_cols = [
    'acteur_id_merge',
    'acteur_id',
    'mandat_id',
    'ident_civ_ANM',           # Civilité (M./Mme)
    'ident_prenom_ANM',        # Prénom
    'ident_nom_ANM',           # Nom
    'ident_male_ANM',          # Sexe (boolean)
    'ident_year_birth_ANM',    # Année naissance
    'profession_libellecourant_ANM',  # Profession
    'profession_PCS_2003_ANM',  # PCS 2003
    'profession_PCS3_2003_ANM', # PCS 2003 detailed
    'profession_PCS_2020_ANM',  # PCS 2020
    'election_lieu_departement_ANM',  # Département
    'election_lieu_numcirco_ANM',     # Numéro circonscription
    'mandat_legislature_ANM',         # Législature du mandat
    'mandat_annee_debut_ANM',   # Année début du mandat
    'mandat_annee_fin_ANM',     # Année fin du mandat
    'groupe_id_ANO',            # Groupe ID
    'groupe_libelle_ANO',       # Groupe libellé
    'groupe_abrege_ANO',        # Groupe abrégé
]

df_mandats_select = df_mandats[[col for col in mandats_cols if col in df_mandats.columns]]

# Remove duplicates sur la clé merge (acteur_id_merge)
df_mandats_select = df_mandats_select.drop_duplicates(subset=['acteur_id_merge'])

print(f"Mandats after selection: {df_mandats_select.shape}")
print(f"Unique acteurs (merge key): {df_mandats_select['acteur_id_merge'].nunique()}")
print(f"Unique mandats: {df_mandats_select['mandat_id'].nunique() if 'mandat_id' in df_mandats_select.columns else 'NA'}")
print(f"Columns selected: {df_mandats_select.columns.tolist()}")


C:\Users\rorod\AppData\Local\Temp\ipykernel_86072\3215096090.py:9: DtypeWarning: Columns (20,21,24,25) have mixed types. Specify dtype option on import or set low_memory=False.
  df_mandats = pd.read_csv(os.path.join(mandats_path, "mandats.csv"))


Loaded mandats (FINAL version with groupe info): (165465, 39)
Mandats columns (first 15): ['organe_id', 'organe_libelle_ANO', 'ident_civ_ANM', 'ident_prenom_ANM', 'ident_nom_ANM', 'ident_alpha_ANM', 'acteur_id', 'ident_datenais_ANM', 'ident_villenais_ANM', 'ident_depnais_ANM', 'ident_paysnais_ANM', 'profession_libellecourant_ANM', 'profession_catsocpro_ANM', 'profession_famsocpro_ANM', 'mandat_type_ANM']


C:\Users\rorod\AppData\Local\Temp\ipykernel_86072\3215096090.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final['acteur_id_merge'] = left_ids_norm


Top chevauchements acteur_id (mandats vs scrutins):
  acteur_id: intersections 1961, couverture 0.0010
  organe_id: intersections 0, couverture 0.0000
  organe_libelle_ANO: intersections 0, couverture 0.0000
  ident_civ_ANM: intersections 0, couverture 0.0000
  ident_prenom_ANM: intersections 0, couverture 0.0000
Colonne retenue pour acteur_id_merge: acteur_id (intersections 1961)
Mandats after selection: (3092, 14)
Unique acteurs (merge key): 3092
Unique mandats: 3092
Columns selected: ['acteur_id_merge', 'acteur_id', 'mandat_id', 'ident_civ_ANM', 'ident_prenom_ANM', 'ident_nom_ANM', 'ident_male_ANM', 'ident_year_birth_ANM', 'profession_libellecourant_ANM', 'election_lieu_departement_ANM', 'election_lieu_numcirco_ANM', 'mandat_legislature_ANM', 'mandat_annee_debut_ANM', 'mandat_annee_fin_ANM']


In [11]:
def drop_dup(df):
    dup_cols = [col for col in df.columns if df.columns.tolist().count(col) > 1]
    dup_cols = list(dict.fromkeys(dup_cols))
    if dup_cols:
        print(f"Duplicated column names: {dup_cols}")
        dup_backup = pd.DataFrame(index=df.index)
        for col in dup_cols:
            col_block = df.loc[:, col]
            if isinstance(col_block, pd.DataFrame):
                dup_backup[col] = col_block.iloc[:, 0]
            else:
                dup_backup[col] = col_block
        df = df.loc[:, ~df.columns.duplicated(keep=False)]
        df = df.merge(dup_backup, left_index=True, right_index=True, how='left')
        print(f"After resolving duplicates: {df.shape}")
        return df
    else:
        print("No duplicated column names detected")
        return df
df_final = drop_dup(df_final)
df_mandats_select = drop_dup(df_mandats_select)

Duplicated column names: ['scrutin_id', 'acteur_id', 'groupe_id']
After resolving duplicates: (2043131, 34)
No duplicated column names detected


In [12]:
# Cell 10: Merge with final dataset
# Merge sur la clé normalisée acteur_id_merge (acteur PA) pour récupérer les infos mandats
# Use left_on/right_on to handle acteur_id naming (avoid duplicate column names)
df_final_with_names = df_final.merge(
    df_mandats_select,
    left_on=['acteur_id_merge'],
    right_on=['acteur_id_merge'],
    how='left',
    suffixes=('', '_mandats')
)

print(f"After merge with mandats: {df_final_with_names.shape}")
print(f"Rows with missing nom: {df_final_with_names['ident_nom_ANM'].isna().sum() if 'ident_nom_ANM' in df_final_with_names.columns else 'colonne absente'}")
print(f"Rows with missing prenom: {df_final_with_names['ident_prenom_ANM'].isna().sum() if 'ident_prenom_ANM' in df_final_with_names.columns else 'colonne absente'}")

# Create full name column (handle NaN values properly)
def create_full_name(row):
    prenom = str(row['ident_prenom_ANM']).strip() if ('ident_prenom_ANM' in row and pd.notna(row['ident_prenom_ANM'])) else ''
    nom = str(row['ident_nom_ANM']).strip() if ('ident_nom_ANM' in row and pd.notna(row['ident_nom_ANM'])) else ''
    full_name = ' '.join([p for p in [prenom, nom] if p])
    return full_name if full_name else None

df_final_with_names['depute_nom_complet'] = df_final_with_names.apply(create_full_name, axis=1)

# Reorder: put deputy info after IDs
id_cols = ['scrutin_id', 'acteur_id', 'acteur_id_merge', 'mandat_id', 'seance_id', 'groupe_id']
deputy_cols = ['depute_nom_complet', 'ident_civ_ANM', 'ident_prenom_ANM', 'ident_nom_ANM',
               'ident_male_ANM', 'ident_year_birth_ANM', 'profession_libellecourant_ANM',
               'election_lieu_departement_ANM', 'election_lieu_numcirco_ANM', 'mandat_legislature_ANM']
deputy_cols = [col for col in deputy_cols if col in df_final_with_names.columns]
other_cols = [col for col in df_final_with_names.columns if col not in id_cols + deputy_cols]

# Conserver uniquement les colonnes qui existent
id_cols = [c for c in id_cols if c in df_final_with_names.columns]

df_final_with_names = df_final_with_names[id_cols + deputy_cols + other_cols]

print(f"\nFinal shape: {df_final_with_names.shape}")


After merge with mandats: (2043131, 47)
Rows with missing nom: 2
Rows with missing prenom: 2

Final shape: (2043131, 48)


In [13]:
# Cell 11: Save final dataset with names
df_final_with_names.to_csv(os.path.join(path3, "scrutins_complete.csv"), index=False)
print(f"Final dataset saved to {path3}/scrutins_complete.csv")

# Display sample
print(f"\nSample rows:")
print(df_final_with_names[['depute_nom_complet', 'acteur_vote', 'groupe_abrege_ANO', 'scrutin_titre']].head(10))

# Summary
print(f"\nSummary:")
print(f"  Total votes: {df_final_with_names.shape[0]:,}")
print(f"  Unique députés: {df_final_with_names['acteur_id'].nunique()}")
print(f"  Unique scrutins: {df_final_with_names['scrutin_id'].nunique()}")
print(f"  Legislatures: {sorted(df_final_with_names['scrutin_legislature'].unique())}")

Final dataset saved to c:\Users\rorod\Documents\Pro\RDB Consulting\Mission Hélène 11-2025--02-2026\DIMIG (collab with Hélène)\AN/Scrutins/Scrutins\3_dataset/scrutins_complete.csv

Sample rows:
   depute_nom_complet acteur_vote groupe_abrege_ANO  \
0   Jean-Marc Ayrault  non votant               SRC   
1    Claude Bartolone  non votant               SRC   
2      Laurent Fabius  non votant               SRC   
3      Alain Vidalies  non votant               SRC   
4       François Lamy  non votant               SRC   
5  Marylise Lebranchu  non votant               SRC   
6  Frédéric Cuvillier  non votant               SRC   
7    Stéphane Le Foll  non votant               SRC   
8        Benoît Hamon  non votant               SRC   
9    Marisol Touraine  non votant               SRC   

                                       scrutin_titre  
0  la déclaration de politique générale du gouver...  
1  la déclaration de politique générale du gouver...  
2  la déclaration de politique génér